In [256]:
import numpy as np 
import pandas as pd 
from matplotlib import pyplot
import seaborn as sns

In [257]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv')
df.sample(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
177,2070-FNEXE,Female,1,No,No,7,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Bank transfer (automatic),76.45,503.6,Yes
5326,5688-KZTSN,Male,0,Yes,Yes,15,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Electronic check,20.00,288.05,Yes
6516,6876-ADESB,Male,0,No,Yes,1,Yes,No,DSL,No,...,Yes,No,No,No,Month-to-month,No,Electronic check,48.95,48.95,Yes
965,9889-TMAHG,Male,1,No,No,8,Yes,No,Fiber optic,Yes,...,No,Yes,Yes,Yes,Month-to-month,No,Credit card (automatic),100.30,832.35,Yes
6194,2868-LLSKM,Female,0,Yes,Yes,68,Yes,Yes,Fiber optic,No,...,No,Yes,No,No,One year,Yes,Bank transfer (automatic),83.65,5733.4,No


In [258]:
# df.info()

In [259]:
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float)

In [260]:
df[df['TotalCharges'].isna()][
    ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']
] 

,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,NaN,No
753,0,20.25,NaN,No
936,0,80.85,NaN,No
1082,0,25.75,NaN,No
1340,0,56.05,NaN,No
3331,0,19.85,NaN,No
3826,0,25.35,NaN,No
4380,0,20.00,NaN,No
5218,0,19.70,NaN,No
6670,0,73.35,NaN,No


In [261]:
df[df['TotalCharges'].isna()].T   # To check these customer are legit and there is no corrupted data

,488,753,936,1082,1340,3331,3826,4380,5218,6670,6754
customerID,4472-LVYGI,3115-CZMZD,5709-LVOEQ,4367-NUYAO,1371-DWPAZ,7644-OMVMY,3213-VVOLG,2520-SGTTA,2923-ARZLG,4075-WKNIU,2775-SEFEE
gender,Female,Male,Female,Male,Female,Male,Male,Female,Male,Female,Male
SeniorCitizen,0,0,0,0,0,0,0,0,0,0,0
Partner,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,No
Dependents,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes,Yes
tenure,0,0,0,0,0,0,0,0,0,0,0
PhoneService,No,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,Yes,Yes
MultipleLines,No phone service,No,No,Yes,No phone service,No,Yes,No,No,Yes,Yes
InternetService,DSL,No,DSL,No,DSL,No,No,No,No,DSL,DSL
OnlineSecurity,Yes,No internet service,Yes,No internet service,Yes,No internet service,No internet service,No internet service,No internet service,No,Yes


In [262]:
df['TotalCharges'].isna().sum()

np.int64(11)

### Handling missing TotalCharges:
11 records had missing TotalCharges. Investigation showed that all 11 customers had tenure = 0, indicating that they were new customers. Since TotalCharges represents accumulated charges, these missing values were replaced with 0 rather than using statistical imputation

In [263]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)  
df['TotalCharges'].isna().sum()

np.int64(0)

### Remvoing "customerID" because this will not effect the output of the final model 

In [264]:
df = df.iloc[:, 1:]
df.shape

(7043, 20)

### Separate features (X) and target (y)

In [265]:
X = df.iloc[:, :-1]
Y = df.iloc[:, -1]

# X.shape 
Y.shape 

(7043,)

### Changing Y values from str (Yes/No) to num (1/0) since logistic reg takes numerical values as input

In [266]:
Y = Y.map({'Yes' : 1, 'No' : 0})
Y.value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

### Splitting the data into training and test

In [267]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2, stratify=Y)


In [268]:
# y_train.value_counts(normalize=True)
# y_test.value_counts(normalize=True)

In [269]:
x_train.dtypes

gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges        float64
dtype: object

In [270]:
numeric_features = x_train.select_dtypes(
    include=['int64', 'float64']
).columns

categorical_features = x_train.select_dtypes(
    include=['object', 'str']
).columns

# print("Numerical:", list(numeric_features))
print("Categorical:", list(categorical_features))

Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


### Building a preprocessing pipeline

In [271]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

For numerical columns → standardize 
For categorical columns → one-hot encode

In [272]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)